In [2]:
TRAINING_FINISHED = False
RANDOM_STATE = 42
MAX_ITERATIONS = 5000
MAX_ITERATIONS_SVM = 1000000
TRAINING_SET_SIZE = 0.7
VALIDATION_TEST_SET_RATIO = 0.5

In [3]:
import pandas as pd

data = pd.read_csv('reviews.csv', sep=",",
                   usecols=['rating', 'review_text', 'helpful'])


In [4]:
data['rating'] = data['rating'].fillna(0)
data['helpful'] = data['helpful'].fillna(0)
data['review_text'] = data['review_text'].fillna("")

In [5]:
# lowercase all text
data['review_text'] = data['review_text'].str.lower()

In [6]:
import contractions

# expand contractions
data['review_text'] = data['review_text'].apply(contractions.fix)

In [7]:
import emoji

# interpret emojis as words
data['review_text'] = data['review_text'].apply(emoji.demojize)

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import csr_matrix, hstack

tfidf_vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2
)
X_text = tfidf_vectorizer.fit_transform(data['review_text'])
X_helpful = csr_matrix(data['helpful'].values.reshape(-1, 1))

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 1271 stored elements and shape (6210, 1)>
  Coords	Values
  (4, 0)	1
  (6, 0)	1
  (12, 0)	1
  (13, 0)	1
  (14, 0)	8
  (15, 0)	1
  (25, 0)	1
  (26, 0)	1
  (28, 0)	1
  (29, 0)	1
  (31, 0)	2
  (34, 0)	1
  (43, 0)	1
  (45, 0)	1
  (51, 0)	1
  (53, 0)	1
  (63, 0)	1
  (67, 0)	1
  (77, 0)	1
  (81, 0)	1
  (86, 0)	2
  (94, 0)	1
  (95, 0)	3
  (98, 0)	1
  (107, 0)	2
  :	:
  (6062, 0)	1
  (6071, 0)	1
  (6073, 0)	1
  (6075, 0)	1
  (6079, 0)	1
  (6080, 0)	2
  (6082, 0)	121
  (6090, 0)	1
  (6091, 0)	14
  (6101, 0)	1
  (6117, 0)	1
  (6119, 0)	1
  (6120, 0)	1
  (6122, 0)	1
  (6126, 0)	11
  (6128, 0)	5
  (6129, 0)	1
  (6131, 0)	1
  (6137, 0)	1
  (6158, 0)	1
  (6160, 0)	4
  (6163, 0)	7
  (6182, 0)	1
  (6190, 0)	1
  (6199, 0)	1
0       0
1       0
2       0
3       0
4       1
       ..
6205    0
6206    0
6207    0
6208    0
6209    0
Name: helpful, Length: 6210, dtype: int64


In [9]:
X = hstack([X_text, X_helpful])
y = data['rating']

In [10]:
from sklearn.model_selection import train_test_split, cross_val_score

# Split base data into 2 sets, 30% for (test + validation), 70% train
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=1 - TRAINING_SET_SIZE, random_state=RANDOM_STATE, stratify=y
)

# Split (test + validation) set into 50% (=15%) test and 50% (=15%) validation
X_validation, X_test, y_validation, y_test = train_test_split(
    X_temp, y_temp, test_size=VALIDATION_TEST_SET_RATIO, random_state=RANDOM_STATE, stratify=y_temp
)

In [11]:
models = []

In [12]:
from sklearn.svm import SVC

# Support Vector Machine model
svc_model = SVC(
    kernel='linear',
    class_weight='balanced',
    random_state=RANDOM_STATE,
    max_iter=MAX_ITERATIONS_SVM
)
svc_model.fit(X_train, y_train)
models.append(svc_model)

In [13]:
from sklearn.linear_model import LogisticRegression

# Logistic Regression model
logreg_model = LogisticRegression(
    class_weight='balanced',
    random_state=RANDOM_STATE,
    max_iter=MAX_ITERATIONS
)
logreg_model.fit(X_train, y_train)
models.append(logreg_model)

In [14]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    random_state=RANDOM_STATE,
    max_depth=9,
    class_weight='balanced',
    n_estimators=300
)
rf_model.fit(X_train, y_train)
models.append(rf_model)

In [15]:
from sklearn.utils import compute_sample_weight
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.neural_network import MLPClassifier

sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

architectures = [
    (64,),
    (128, 64),
    (256, 128),
    (128, 64, 32),
    (128, 128, 64),
    (128, 128, 32),
]
best_score = 0
best_mlp = None

for i, arch in enumerate(architectures):
    print(f"Training architecture #{i} {arch}:")
    mlp = MLPClassifier(
        hidden_layer_sizes=arch,
        activation='relu',
        max_iter=MAX_ITERATIONS,
        random_state=RANDOM_STATE,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=5
    )
    mlp.fit(X_train, y_train, sample_weight=sample_weights)

    val_score = accuracy_score(y_validation, mlp.predict(X_validation))
    print(f"- Validation Accuracy: {val_score:.4f}")

    if val_score > best_score:
        best_score = val_score
        best_mlp = mlp

models.append(best_mlp)

Training architecture #0 (64,):
- Validation Accuracy: 0.5311
Training architecture #1 (128, 64):
- Validation Accuracy: 0.5451
Training architecture #2 (256, 128):
- Validation Accuracy: 0.5805
Training architecture #3 (128, 64, 32):
- Validation Accuracy: 0.4979
Training architecture #4 (128, 128, 64):
- Validation Accuracy: 0.5322
Training architecture #5 (128, 128, 32):
- Validation Accuracy: 0.4217


In [16]:
from sklearn.metrics import accuracy_score, classification_report

# PLOT THAT instead
for i, model in enumerate(models):
    print(f'Evaluate Model #{i}: {type(model).__name__}')
    # SVC uses scaled data
    y_pred = model.predict(X_validation)
    print(f'Accuracy: {accuracy_score(y_validation, y_pred):.4f}')
    print(classification_report(y_validation, y_pred))

    # print("Cross-validation scores: {}".format(cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')))
    # print("Confusion matrix:\n{}".format(confusion_matrix(y_validation, y_pred)))


Evaluate Model #0: SVC
Accuracy: 0.4850
              precision    recall  f1-score   support

           1       0.48      0.49      0.48       237
           2       0.12      0.10      0.11        70
           3       0.18      0.33      0.23        76
           4       0.07      0.05      0.06        95
           5       0.71      0.66      0.68       454

    accuracy                           0.48       932
   macro avg       0.31      0.33      0.31       932
weighted avg       0.50      0.48      0.49       932

Confusion matrix:
[[115  17  30  15  60]
 [ 25   7  18   6  14]
 [ 27   6  25   2  16]
 [ 22   5  29   5  34]
 [ 53  21  35  45 300]]
Evaluate Model #1: LogisticRegression
Accuracy: 0.4785
              precision    recall  f1-score   support

           1       0.53      0.57      0.55       237
           2       0.14      0.19      0.16        70
           3       0.17      0.37      0.23        76
           4       0.11      0.11      0.11        95
           

In [17]:
# FINAL SCORING; ONLY USE AFTER FINISHED TRAINING
if not TRAINING_FINISHED:
    quit(0)
for i, model in enumerate(models):
    print(f'Evaluate Model #{i} with type: {type(model).__name__}')
    y_pred = model.predict(X_test)
    print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
    print(classification_report(y_test, y_pred))

Evaluate Model #0 with type: SVC
Accuracy: 0.4785
              precision    recall  f1-score   support

           1       0.60      0.46      0.52       238
           2       0.12      0.09      0.10        70
           3       0.14      0.26      0.18        76
           4       0.05      0.03      0.04        95
           5       0.63      0.68      0.66       453

    accuracy                           0.48       932
   macro avg       0.31      0.30      0.30       932
weighted avg       0.48      0.48      0.48       932

Evaluate Model #1 with type: LogisticRegression
Accuracy: 0.4571
              precision    recall  f1-score   support

           1       0.54      0.57      0.56       238
           2       0.12      0.14      0.13        70
           3       0.13      0.32      0.18        76
           4       0.09      0.09      0.09        95
           5       0.78      0.55      0.64       453

    accuracy                           0.46       932
   macro avg    